# Inspecting Other and Bad example documents

A small random sample of training documents whose **most probable label** in the Qwen3.8-27B
soft labels is a chosen label (by default one of the two labels WebOrganizer does not have,
*Other* and *Bad example*), for reading by hand.

**Data.** `results/statistics/other_bad_examples/<language>.jsonl`, written by
`scripts/data_analysis/extract_top_label_documents.py` from the training split of
`results/datasets/{topic,format}/hplt-finetune`. A file holds every training document of
that language whose top topic label *or* top format label is Other or Bad example, with
the top two labels and probabilities of both label types, the source bucket (clean, noisy
or poor), the URL and the text.

**How to use it.** Set `LANGUAGE`, `LABEL_TYPE` and `LABEL` in the configuration cell and
run all cells. Change `SEED` for a different sample. `SOURCE` limits the sample to one
quality bucket.

**Other languages.** Language names are the dataset's (lower case, underscores for spaces,
e.g. `northern_sotho`). If the file for `LANGUAGE` doesn't exist yet, the next cell stops
and shows the extraction command. With `EXTRACT_MISSING = True` it runs the extraction
itself instead, which takes about 5 minutes over sshfs because it scans the training split.
Other labels than Other and Bad example need a new extraction with `--labels` and a different
`DATA_DIR`.

In [11]:
from __future__ import annotations

import html
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# What to sample. LABEL_TYPE: "topic" or "format". LABEL: a label extracted into DATA_DIR
# (by default "Other" or "Bad example"). SOURCE: None for all buckets, or "clean_random",
# "noisy_random" or "poor_random".
LANGUAGE = "finnish"
LABEL_TYPE = "format"
LABEL = "Other"
SOURCE = None
SAMPLE_SIZE = 10
SEED = 0
# Characters of each text shown before the "show full text" fold.
PREVIEW_CHARS = 1500
# Run the extraction for LANGUAGE if its file is missing (about 5 minutes).
EXTRACT_MISSING = False

DATA_DIR = REPO / "results/statistics/other_bad_examples"
EXTRACTED_LABELS = ("Other", "Bad example")
SCRIPT = REPO / "scripts/data_analysis/extract_top_label_documents.py"
DATASETS = {label_type: REPO / f"results/datasets/{label_type}/hplt-finetune" for label_type in ("topic", "format")}

if LABEL_TYPE not in DATASETS:
    raise ValueError(f"LABEL_TYPE must be 'topic' or 'format', not {LABEL_TYPE!r}")
if DATA_DIR.name == "other_bad_examples" and LABEL not in EXTRACTED_LABELS:
    raise ValueError(f"{DATA_DIR} holds only {EXTRACTED_LABELS}; extract {LABEL!r} with --labels into another DATA_DIR")

## 1. Load the language's documents

Lists the languages already extracted, and extracts `LANGUAGE` first if it is missing and
`EXTRACT_MISSING` is set.

In [12]:
path = DATA_DIR / f"{LANGUAGE}.jsonl"
available = sorted(p.stem for p in DATA_DIR.glob("*.jsonl"))
print("Extracted languages:", ", ".join(available) or "none")

command = [
    sys.executable, str(SCRIPT),
    "--topic_dataset", str(DATASETS["topic"]),
    "--format_dataset", str(DATASETS["format"]),
    "--languages", LANGUAGE,
    "--labels", *EXTRACTED_LABELS,
    "--output_dir", str(DATA_DIR),
]
if not path.exists():
    if not EXTRACT_MISSING:
        raise FileNotFoundError(
            f"No {path}. Set EXTRACT_MISSING = True, or run from the repository root:\n  "
            + " ".join(f"'{part}'" if " " in part else part for part in command)
        )
    print("Extracting", LANGUAGE, "...")
    subprocess.run(command, check=True)

records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
if not records:
    raise ValueError(f"{path} is empty")
docs = pd.json_normalize(records, sep="_")
print(f"{len(docs):,} {LANGUAGE} training documents with a top topic or format label in {EXTRACTED_LABELS}")

Extracted languages: english, finnish
3,760 finnish training documents with a top topic or format label in ('Other', 'Bad example')


## 2. How many there are

Documents per top label and source bucket, for both label types. A document can appear
under both, e.g. Bad example for topic and Spam / Ads or Bad example for format.

In [13]:
counts = []
for label_type in ("topic", "format"):
    top = docs[f"{label_type}_top"]
    chosen = docs[top.isin(EXTRACTED_LABELS)]
    table = pd.crosstab(chosen[f"{label_type}_top"], chosen["source"], margins=True, margins_name="all")
    table.index = [f"{label_type}: {label}" for label in table.index]
    counts.append(table)
display(pd.concat(counts).fillna(0).astype(int))

source,clean_random,noisy_random,poor_random,all
topic: Bad example,168,262,753,1183
topic: Other,123,190,82,395
topic: all,291,452,835,1578
format: Bad example,41,521,2026,2588
format: Other,41,87,45,173
format: all,82,608,2071,2761


## 3. The other label type of the chosen documents

For the documents whose top `LABEL_TYPE` label is `LABEL`: what the *other* label type says,
and what the runner-up label of `LABEL_TYPE` is. The runner-up shows what the page would
have been labelled without Other or Bad example.

In [14]:
other_type = "format" if LABEL_TYPE == "topic" else "topic"
selected = docs[docs[f"{LABEL_TYPE}_top"] == LABEL]
if SOURCE is not None:
    if SOURCE not in set(docs["source"]):
        raise ValueError(f"SOURCE {SOURCE!r} not in {sorted(set(docs['source']))}")
    selected = selected[selected["source"] == SOURCE]
if selected.empty:
    raise ValueError(f"No {LANGUAGE} documents with top {LABEL_TYPE} label {LABEL!r} (source {SOURCE})")

where = f", source {SOURCE}" if SOURCE else ""
display(Markdown(f"**{len(selected):,} {LANGUAGE} documents with top {LABEL_TYPE} label {LABEL}{where}.** "
                 f"Median probability of {LABEL}: {selected[f'{LABEL_TYPE}_top_prob'].median():.2f}."))
# Two independent rankings side by side (row i is the i-th most common of each, not the same documents).
rankings = {
    f"top {other_type} label": selected[f"{other_type}_top"],
    f"runner-up {LABEL_TYPE} label": selected[f"{LABEL_TYPE}_second"],
}
summary = pd.concat(
    {
        name: values.value_counts().head(10).rename_axis("label").reset_index(name="documents")
        for name, values in rankings.items()
    },
    axis=1,
)
summary.index = range(1, len(summary) + 1)
display(summary)

**173 finnish documents with top format label Other.** Median probability of Other: 0.48.

top topic label           runner-up format label          
                 label documents                  label documents
1             Religion        28           About (Org.)        23
2   Finance & Business        28       Customer Support        20
3       Home & Hobbies        16        Structured Data        17
4             Politics        13            Bad example        17
5     Sports & Fitness        11           Product Page        15
6     Education & Jobs         9       Creative Writing        15
7               Health         8            News (Org.)        14
8          Social Life         6      Knowledge Article        11
9        Entertainment         6              Truncated         8
10      Transportation         6        Comment Section         7

## 4. The sample

`SAMPLE_SIZE` documents drawn at random (with `SEED`) from the selection above. Each card
shows the labels of both types with their probabilities, the source bucket and the URL,
then the start of the text; the rest is behind *show full text*.

In [15]:
sample = selected.sample(n=min(SAMPLE_SIZE, len(selected)), random_state=SEED)

# Explicit colours on every element so the cards stay readable in dark editor themes.
CARD = "border:1px solid #c8c8c8;border-radius:6px;padding:10px 14px;margin:12px 0;background:#ffffff;color:#1a1a1a;"
META = "font-size:0.9em;color:#3a3a3a;background:#ffffff;margin:2px 0;"
TEXT = ("white-space:pre-wrap;font-family:monospace;font-size:0.85em;color:#1a1a1a;background:#f6f6f6;"
        "padding:8px;border-radius:4px;margin:6px 0 0 0;")


def label_line(row: pd.Series, label_type: str) -> str:
    # "Bad example 0.998 (then Religion 0.001)", the top label in bold.
    top = html.escape(row[f"{label_type}_top"])
    second = html.escape(row[f"{label_type}_second"])
    return (f"<b>{top}</b> {row[f'{label_type}_top_prob']:.3f} "
            f"(then {second} {row[f'{label_type}_second_prob']:.3f})")


def card(number: int, row: pd.Series) -> str:
    text = row["text"]
    preview, rest = text[:PREVIEW_CHARS], text[PREVIEW_CHARS:]
    url = html.escape(row["url"])
    parts = [
        f'<div style="{CARD}">',
        f'<div style="{META}"><b style="color:#1a1a1a;">{number}. {html.escape(row["id"])}</b> · {row["source"]} · '
        f"{len(text):,} characters</div>",
        f'<div style="{META}">Topic: {label_line(row, "topic")}</div>',
        f'<div style="{META}">Format: {label_line(row, "format")}</div>',
        f'<div style="{META}">URL: <a style="color:#1f5fa8;" href="{url}">{url}</a></div>',
        f'<pre style="{TEXT}">{html.escape(preview)}</pre>',
    ]
    if rest:
        parts.append(
            f'<details style="color:#1a1a1a;background:#ffffff;"><summary style="color:#1f5fa8;cursor:pointer;">'
            f"show full text ({len(rest):,} more characters)</summary>"
            f'<pre style="{TEXT}">{html.escape(rest)}</pre></details>'
        )
    parts.append("</div>")
    return "".join(parts)


display(HTML("".join(card(i, row) for i, (_, row) in enumerate(sample.iterrows(), start=1))))

## 5. The sample as a table

The same documents in one table, for copying IDs or noting a verdict per document.

In [16]:
columns = ["id", "source", "topic_top", "topic_top_prob", "topic_second", "format_top", "format_top_prob",
           "format_second", "url"]
display(sample[columns].reset_index(drop=True))

,id,source,topic_top,topic_top_prob,topic_second,format_top,format_top_prob,format_second,url
0,602eb65cc07efb8246390e3781c0e727,noisy_random,Fashion & Beauty,0.9961,Home & Hobbies,Other,0.5288,Spam / Ads,https://kostyymi.nuorisoseurat.fi/ostetaan-ete...
1,1db2aacd4f871cee508dac799ecad6bc,poor_random,History,0.7654,Other,Other,0.4803,Structured Data,https://time.is/fi/Elbasan
2,cec9859bbec1b4d07ed4bac532702dcb,noisy_random,Education & Jobs,0.3148,Other,Other,0.4164,Bad example,https://www.hiihtomuseo.fi/palautekysely/
3,22bf280613c9789c390e6f3368a0c32e,noisy_random,Sports & Fitness,0.9362,Fashion & Beauty,Other,0.3875,Spam / Ads,http://www.relaa.com/keskustelu/index.php?topi...
4,0609a73e5e7c22880d5f9171bd7bf102,noisy_random,Religion,0.9998,Art & Design,Other,0.4703,Creative Writing,http://pt.youversion.com/bible/verse/finpr/jud...
5,59e889f40a531763de8ab38f967aa427,noisy_random,Transportation,0.9091,Industrial,Other,0.4347,Comment Section,https://status.open311.org/cities/turku/reques...
6,f87f9633efabf9a085932cb2a3aa84a0,clean_random,Religion,0.9997,Finance & Business,Other,0.4617,Creative Writing,http://studybible.info/topics/Finnish/dealing%...
7,139cc0411117785964f7941451f41bed,poor_random,Sports & Fitness,0.9999,Entertainment,Other,0.3177,News (Org.),https://crossfitporvoo.com/vod/wod-29-5-2019/
8,3f0eecb3951c473a482bca57848c8c5c,noisy_random,Health,0.9998,Software,Other,0.4968,Customer Support,http://www.kiropraktikko.org/yhteystiedot/
9,e13bbd2a1dc0a994495cafbcf57a11a7,noisy_random,Entertainment,0.9923,Literature,Other,0.7560,Listicle,http://hpfanit.vuotis.net/joulukalenteri2007/1...
